# 07 — Reconciliation of the LR-only results between the first submission and the revision (CPU)

Answers Reviewer 3, point 1 (and the software/embedding part of point 3) of the second review round (Paper ID 20266676).

**Question.** Table 3 of the first submission and of the revision report different LR-only SemEval scores (XLM-R LR 0.3966 vs 0.3892; IndoBERT LR 0.3702 vs 0.3679), although §4.2 and the README stated that LR results on XLM-R and IndoBERT reproduce the earlier run exactly. That statement was based only on a Tokopedia, seed-42 check (`determinism_check.csv`).

**What this notebook does** (same code as notebook 03; only the procedure switch is varied):

| Block | Content | Output |
|---|---|---|
| A | Embedding fingerprints for every cache (SemEval and Tokopedia); optional fresh re-encoding of SemEval to compare with the cache | `embedding_fingerprints.csv`, `embedding_fresh_vs_cache.csv` |
| B | SemEval, LR-only (XLM-R, IndoBERT, E5), 10 seeds, **two procedures**: inner folds stratified on emotions only (first submission) and on emotions + language (revision). Every run is compared text by text with the stored predictions of both submissions | `B_semeval_runs.csv`, `B_semeval_vs_reference.csv` |
| C | Tokopedia, LR-only, 10 seeds, run twice in this session; compared with both submissions, with a diagnosis of every mismatch (probability drift vs. tie in the class-prior search) | `C_tokopedia_runs.csv`, `C_tokopedia_vs_reference.csv` |
| D | Cell-by-cell reconciliation of Table 3 (LR rows) and per-seed table | `D_table3_LR_reconciliation.csv`, `D_table3_LR_per_seed.csv`, `D_summary.md` |

**Reference files** are read from the public repository at the immutable commit `2c997704beeb8b1c86e1539146864071754ef3bd` (cloned automatically when Internet is on), or from an attached copy of the repository: `results/main/` (first submission: per-seed CSVs, `predictions.zip`, `environment.json`, split manifest) and `results/rerun_frozen/` (revision).

**Before running on Kaggle**
1. *Add Input*: SemEval-2025 Task 11 data, PRDECT-ID, and the **embedding-cache dataset used by notebook 03** (`embedding-cache-me-xbal`). Without the cache the embeddings are recomputed and the comparison loses its meaning.
2. *Settings*: Accelerator **None (CPU)**, Internet **On** (to clone the repository at the pinned commit).
3. *Save Version → Save & Run All*. Expected run time ≈ 40–70 min. Download `reconcile_lr_outputs.zip` from the output.


In [ ]:
# ============================================================
# 0. INSTALL & IMPORTS
# ============================================================
!pip -q install iterative-stratification

import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from scipy import stats
import joblib

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
import xgboost as xgb
import transformers
from transformers import AutoTokenizer, AutoModel
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit, MultilabelStratifiedKFold

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOTEBOOK_T0 = time.time()
print("Python", platform.python_version(), "| sklearn", sklearn.__version__, "| xgboost", xgb.__version__,
      "| numpy", np.__version__, "| CPUs", os.cpu_count(), "| device", DEVICE)

## 1. Configuration and reference repository

In [ ]:
# ============================================================
# 1. CONFIG (constants identical to notebook 03; only the switches and paths are new)
# ============================================================
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))

SEEDS        = [42, 7, 13, 77, 101, 202, 303, 404, 505, 606]
SEED_PRIMARY = 42
VAL_FRAC, TEST_FRAC = 0.15, 0.20
N_OOF_FOLDS = 5

FOLD_STRATIFY_BY_LANG = True      # revision default; block B also runs False (= first submission)
W_OBJECTIVE           = "pooled"
XGB_NTHREAD           = 4
XGB_INPUT_DECIMALS    = 5         # irrelevant for the LR-only models reconciled here

LR_MODELS = ["XLM-R LR", "IndoBERT LR", "E5 LR"]
PROCEDURES = {"first_submission_folds": False,     # inner folds stratified on the emotions only
              "revision_folds": True}              # inner folds stratified on the emotions + language

RUN_FINGERPRINTS     = True
RUN_FRESH_SEMEVAL    = True       # re-encode SemEval from the Hugging Face models and compare with the cache (~10-20 min CPU)
RUN_SEMEVAL          = True
RUN_TOKOPEDIA        = True
TOK_REPEATS          = 2          # run every Tokopedia model twice in this session

ENCODERS = {
    "e5":       dict(name="intfloat/multilingual-e5-base", prefix="query: "),
    "xlmr":     dict(name="xlm-roberta-base",              prefix=None),
    "indobert": dict(name="indobenchmark/indobert-base-p1", prefix=None),
}
MAIN_ENCODER = "e5"
MAX_LEN, EMBED_BATCH = 128, 16

WEIGHT_GRID    = np.arange(0.0, 1.0001, 0.05)
THRESHOLD_GRID = np.arange(0.15, 0.8501, 0.01)
MIN_PRECISION, MAX_PPR_RATIO, TIE_TOL = 0.20, 3.0, 0.005
ALPHA_GRID, ALPHA_PASSES = np.arange(0.50, 3.0001, 0.05), 3

XGB_COMMON = dict(max_depth=3, min_child_weight=5, learning_rate=0.05, n_estimators=200,
                  subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.5,
                  tree_method="hist", verbosity=0)
XGB_RUNTIME = {"n_jobs": XGB_NTHREAD}
XGB_PARAMS_BINARY = dict(objective="binary:logistic", eval_metric="logloss", **XGB_COMMON)
XGB_PARAMS_MULTI  = dict(objective="multi:softprob", eval_metric="mlogloss", **XGB_COMMON)

if SMOKE_TEST:
    SEEDS = [42, 7]; RUN_FRESH_SEMEVAL = False

# ---- paths ------------------------------------------------------
INPUT_ROOTS = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))

def _find(pattern, roots=INPUT_ROOTS):
    for r in roots:
        hits = sorted(glob.glob(os.path.join(r, "**", pattern), recursive=True))
        if hits: return hits[0]
    return None

def _semeval_root():
    hits = [h for r in INPUT_ROOTS for h in sorted(glob.glob(os.path.join(r, "**", "track_a/train/sun.csv"), recursive=True))]
    hits.sort(key=lambda h: "public_data_test" not in h)
    for h in hits:
        root = Path(h).parents[2]
        need = [root / "track_a/dev/sun.csv", root / "track_c/dev/jav.csv", root / "track_c/dev/ind.csv"]
        if all(f.exists() for f in need) and all({"anger", "joy"} <= set(pd.read_csv(f, nrows=1).columns) for f in need):
            return str(root)
    return None

SEMEVAL_ROOT = _semeval_root()
assert SEMEVAL_ROOT, "SemEval-2025 Task 11 data (with labelled dev files) not found: add the dataset as input."
TOKOPEDIA_CSV = _find("PRDECT-ID Dataset.csv") or _find("*PRDECT*.csv") or _find("Tokopedia Review.csv")
assert TOKOPEDIA_CSV, "PRDECT-ID (Tokopedia) CSV not found: add the dataset as input."

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT  = os.path.join(WORK, "reconcile_lr")
CACHE_OUT = os.path.join(WORK, "embedding_cache")
_cache_hit = _find("tokopedia_e5.npz")
CACHE_IN  = str(Path(_cache_hit).parent) if _cache_hit else None
for d in [OUT_ROOT, CACHE_OUT, f"{OUT_ROOT}/predictions"]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ---- reference results of both submissions (immutable commit) -----
REF_COMMIT = "2c997704beeb8b1c86e1539146864071754ef3bd"
REF_URL    = "https://github.com/wagino89/ijies-low-resource-emotion-classification.git"
REPO_DIR   = os.path.join(WORK, "ref_repo")

def repo_root():
    """Attached copy of the repository (any folder containing results/main/SemEval_final_raw.csv) or a clone at REF_COMMIT."""
    for r in INPUT_ROOTS:
        for h in sorted(glob.glob(os.path.join(r, "**", "results/main/SemEval_final_raw.csv"), recursive=True)):
            root = str(Path(h).parents[2])
            head = subprocess.run(["git", "-C", root, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
            return root, (f"attached input, git {head}" if head else "attached input (no .git: commit not verifiable)")
    if not os.path.exists(f"{REPO_DIR}/.git"):
        cmd = f"git init -q {REPO_DIR} && cd {REPO_DIR} && git remote add origin {REF_URL} && git fetch -q --depth 1 origin {REF_COMMIT} && git checkout -q FETCH_HEAD"
        assert subprocess.run(["bash", "-c", cmd]).returncode == 0, "Clone failed: turn Internet on or attach the repository as input."
    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head == REF_COMMIT, head
    return REPO_DIR, f"git {head}"

REPO, REPO_SOURCE = repo_root()
OLD_DIR, NEW_DIR = f"{REPO}/results/main", f"{REPO}/results/rerun_frozen"
print("SemEval root :", SEMEVAL_ROOT); print("Tokopedia csv:", TOKOPEDIA_CSV)
print("Cache input  :", CACHE_IN);     print("Reference    :", REPO, "|", REPO_SOURCE)
print("Output       :", OUT_ROOT, "| SMOKE_TEST =", SMOKE_TEST)
if not CACHE_IN: print("WARNING: embedding cache not attached -> embeddings are recomputed; reconciliation is then only indicative.")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def append_csv(path, rows):
    new = pd.DataFrame(rows)
    old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True); df.to_csv(path, index=False); return df

def done_keys(path, cols):
    if not os.path.exists(path): return set()
    df = pd.read_csv(path)
    return set(map(tuple, df[cols].astype(str).values.tolist()))

safe = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

def cpu_info():
    try: return subprocess.run(["bash", "-c", "lscpu | grep -E 'Model name|^CPU\\(s\\)'"], capture_output=True, text=True).stdout.strip()
    except Exception: return platform.processor()


## 2. Data and splits (checked against both submissions)

In [ ]:
# ============================================================
# 2. DATA + SPLITS (identical to notebooks 01 and 03) and check against both stored split manifests
# ============================================================
SEMEVAL_LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LANGS = ["sun", "jav", "ind"]          # pooling order of the development partitions

def load_semeval():
    sun = pd.concat([pd.read_csv(f"{SEMEVAL_ROOT}/track_a/train/sun.csv"),
                     pd.read_csv(f"{SEMEVAL_ROOT}/track_a/dev/sun.csv")], ignore_index=True)
    jav = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/jav.csv")
    ind = pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/ind.csv")
    data = {"sun": sun.reset_index(drop=True), "jav": jav.reset_index(drop=True), "ind": ind.reset_index(drop=True)}
    for lang, df in data.items():
        assert "text" in df.columns and all(c in df.columns for c in SEMEVAL_LABELS), lang
        df["uid"] = [f"{lang}_{i:05d}" for i in range(len(df))]
    return data

def load_tokopedia():
    df = pd.read_csv(TOKOPEDIA_CSV)[["Customer Review", "Emotion"]].dropna().reset_index(drop=True)
    le = LabelEncoder(); df["label"] = le.fit_transform(df["Emotion"])
    df["uid"] = [f"tok_{i:05d}" for i in range(len(df))]
    return df, list(le.classes_)

def _two_stage(splitter, y, seed):
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])

SEMEVAL_DATA = load_semeval()
TOKO_DF, TOKO_CLASSES = load_tokopedia()
K_TOK = len(TOKO_CLASSES)
SEMEVAL_SPLITS = {(l, s): _two_stage(MultilabelStratifiedShuffleSplit, SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int), s)
                  for l in LANGS for s in SEEDS}
TOKO_SPLITS = {s: _two_stage(StratifiedShuffleSplit, TOKO_DF["label"].values, s) for s in SEEDS}

man = []
for (l, s), parts in SEMEVAL_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        man += [{"dataset": "SemEval", "lang": l, "seed": s, "split": sp, "uid": u} for u in SEMEVAL_DATA[l]["uid"].values[idx]]
for s, parts in TOKO_SPLITS.items():
    for sp, idx in zip(["train", "val", "test"], parts):
        man += [{"dataset": "Tokopedia", "lang": "ind", "seed": s, "split": sp, "uid": u} for u in TOKO_DF["uid"].values[idx]]
MAN = pd.DataFrame(man); MAN.to_csv(f"{OUT_ROOT}/split_manifest_this_run.csv", index=False)

def _key(df): return set(map(tuple, df[["dataset", "lang", "seed", "split", "uid"]].astype(str).values.tolist()))
split_check = []
for tag, path in [("first submission", f"{OLD_DIR}/split_manifest_all.csv"), ("revision", f"{NEW_DIR}/split_manifest_all.csv")]:
    ref = pd.read_csv(path); ref = ref[ref.seed.isin(SEEDS)]
    split_check.append({"reference": tag, "rows_reference": len(ref), "rows_this_run": len(MAN), "identical": _key(ref) == _key(MAN),
                        "sha256_reference_file": hashlib.sha256(open(path, "rb").read()).hexdigest()})
SPLIT_CHECK = pd.DataFrame(split_check); SPLIT_CHECK.to_csv(f"{OUT_ROOT}/split_check.csv", index=False)
print(SPLIT_CHECK.to_string(index=False))
assert SPLIT_CHECK.identical.all(), "Split manifests differ: stop and investigate before reconciling scores."
print("SemEval:", {l: len(d) for l, d in SEMEVAL_DATA.items()}, "| Tokopedia:", len(TOKO_DF), TOKO_CLASSES)


## 3. Embeddings — Block A: fingerprints and fresh re-encoding

In [ ]:
# ============================================================
# 3. EMBEDDINGS (identical encoding; reused from the cache when attached)
# ============================================================
def masked_mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(texts, model, tok, prefix=None, batch=EMBED_BATCH):
    texts = [(prefix or "") + str(x) for x in texts]; rows = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i + batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        z = masked_mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        rows.append(F.normalize(z, p=2, dim=1).cpu().numpy())
    return np.vstack(rows)

def cached_embeddings(cache_name, texts, enc_key):
    for root in [CACHE_IN, CACHE_OUT]:
        if root and os.path.exists(f"{root}/{cache_name}.npz"):
            emb = np.load(f"{root}/{cache_name}.npz")["embedding"]
            if emb.shape[0] == len(texts):
                print("[cache]", cache_name, emb.shape); return emb
    print("[computing on", DEVICE, "]", cache_name, "— this is slow on CPU; attach the embedding cache if possible")
    cfg = ENCODERS[enc_key]
    tok = AutoTokenizer.from_pretrained(cfg["name"]); model = AutoModel.from_pretrained(cfg["name"]).to(DEVICE).eval()
    emb = encode(texts, model, tok, cfg["prefix"])
    np.savez_compressed(f"{CACHE_OUT}/{cache_name}.npz", embedding=emb)
    del model, tok; gc.collect(); return emb

SEMEVAL_EMB = {k: {l: cached_embeddings(f"semeval_{k}_{l}", SEMEVAL_DATA[l]["text"].tolist(), k) for l in LANGS} for k in ENCODERS}
TOKO_EMB = {k: cached_embeddings(f"tokopedia_{k}", TOKO_DF["Customer Review"].tolist(), k) for k in ENCODERS}

# ---- Block A: fingerprints of every embedding matrix (same hash as notebook 03) ----
fp = lambda v: hashlib.md5(np.round(v, 6).astype(np.float32).tobytes()).hexdigest()[:12]
EMB_FINGERPRINT = {f"tokopedia_{k}": fp(v) for k, v in TOKO_EMB.items()}
EMB_FINGERPRINT.update({f"semeval_{k}_{l}": fp(SEMEVAL_EMB[k][l]) for k in ENCODERS for l in LANGS})
REV_FP = json.load(open(f"{NEW_DIR}/environment_cpu.json")).get("embedding_fingerprints", {})
fp_rows = [{"matrix": m, "fingerprint_this_run": h, "fingerprint_revision": REV_FP.get(m, "not recorded"),
            "identical_to_revision": (REV_FP.get(m) == h) if m in REV_FP else None,
            "source": "cache" if CACHE_IN and os.path.exists(f"{CACHE_IN}/{m}.npz") else "computed"} for m, h in EMB_FINGERPRINT.items()]
pd.DataFrame(fp_rows).to_csv(f"{OUT_ROOT}/embedding_fingerprints.csv", index=False)
print(pd.DataFrame(fp_rows).to_string(index=False))

# optional: re-encode SemEval now and compare with the cached matrices (shows the size of hardware/library drift)
FRESH_CSV = f"{OUT_ROOT}/embedding_fresh_vs_cache.csv"
if RUN_FRESH_SEMEVAL and CACHE_IN and not os.path.exists(FRESH_CSV):
    rows = []
    for k, cfg in ENCODERS.items():
        tok = AutoTokenizer.from_pretrained(cfg["name"]); model = AutoModel.from_pretrained(cfg["name"]).to(DEVICE).eval()
        for l in LANGS:
            fresh = encode(SEMEVAL_DATA[l]["text"].tolist(), model, tok, cfg["prefix"]); cached = SEMEVAL_EMB[k][l]
            rows.append({"encoder": k, "lang": l, "n": len(fresh), "max_abs_diff": float(np.abs(fresh - cached).max()),
                         "min_cosine": float(np.min(np.sum(fresh * cached, 1))), "bit_identical": bool(np.array_equal(fresh, cached)),
                         "fingerprint_fresh": fp(fresh), "fingerprint_cache": fp(cached)})
            print(rows[-1])
        del model, tok; gc.collect()
    pd.DataFrame(rows).to_csv(FRESH_CSV, index=False)


## 4. Classifier engines (verbatim from notebook 03)

In [ ]:
# ============================================================
# 4. ENGINES
# ============================================================
def fit_preprocessor(X, seed=42):
    sc = StandardScaler().fit(X)
    pc = PCA(n_components=0.95, random_state=seed).fit(sc.transform(X))
    return (lambda A: pc.transform(sc.transform(A))), sc, pc

def xgb_in(Z):
    return np.round(Z, XGB_INPUT_DECIMALS) if XGB_INPUT_DECIMALS is not None else Z

def new_log():
    return {"constant_fits": 0, "threshold_fallbacks": 0}

class ConstantBinary:
    # used when an emotion has no positive (or no negative) example in a training fold
    def __init__(self, p): self.p = float(p)
    def predict_proba(self, X): return np.c_[np.full(len(X), 1 - self.p), np.full(len(X), self.p)]

def combine(px, pl, w):
    if px is None: return pl
    if pl is None: return px
    return w * px + (1 - w) * pl

# ---------------- multi-label (SemEval) ----------------
def train_ml(X, y, cfg, seed, log=None):
    xm = {} if cfg["use_xgb"] else None; lm = {} if cfg["use_lr"] else None
    for j, e in enumerate(SEMEVAL_LABELS):
        yy = y[:, j]; pos = int(yy.sum()); neg = len(yy) - pos
        degenerate = pos == 0 or neg == 0
        if degenerate and log is not None: log["constant_fits"] += 1
        if cfg["use_xgb"]:
            if degenerate: xm[e] = ConstantBinary(yy.mean())
            else:
                ratio = neg / pos if cfg["cost_sensitive"] else 1.0
                xm[e] = xgb.XGBClassifier(scale_pos_weight=ratio, random_state=seed, n_jobs=XGB_RUNTIME["n_jobs"],
                                          **XGB_PARAMS_BINARY).fit(xgb_in(X), yy)
        if cfg["use_lr"]:
            if degenerate: lm[e] = ConstantBinary(yy.mean())
            else:
                lm[e] = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                           class_weight="balanced" if cfg["lr_balanced"] else None,
                                           random_state=seed).fit(X, yy)
    return xm, lm

def proba_ml(X, xm, lm):
    px = np.column_stack([xm[e].predict_proba(xgb_in(X))[:, 1] for e in SEMEVAL_LABELS]) if xm is not None else None
    pl = np.column_stack([lm[e].predict_proba(X)[:, 1] for e in SEMEVAL_LABELS]) if lm is not None else None
    return px, pl

def ml_score(y, yp, groups=None, objective="pooled"):
    if objective == "pooled" or groups is None:
        return f1_score(y, yp, average="macro", zero_division=0)
    return float(np.mean([f1_score(y[groups == g], yp[groups == g], average="macro", zero_division=0) for g in np.unique(groups)]))

def tune_w_ml(px, pl, y, groups=None, objective="pooled"):
    # ensemble weight at a fixed 0.5 threshold; ties -> smallest weight (first maximum)
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = ml_score(y, (combine(px, pl, w) >= 0.5).astype(int), groups, objective)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_th(p, y, log=None):
    # one global threshold per emotion; constraints: precision >= 0.20 and positive rate <= max(3*base, base+0.05);
    # ties within 0.005 -> highest threshold; if no threshold satisfies the constraints -> unconstrained best F1
    out = []
    for j in range(p.shape[1]):
        base = float(y[:, j].mean()); cap = min(1.0, max(MAX_PPR_RATIO * base, base + 0.05)); sc = []
        for t in THRESHOLD_GRID:
            yp = (p[:, j] >= t).astype(int)
            if yp.mean() > cap: sc.append(-np.inf); continue
            prec = y[:, j][yp == 1].mean() if yp.sum() else 0
            if prec < MIN_PRECISION: sc.append(-np.inf); continue
            sc.append(f1_score(y[:, j], yp, zero_division=0))
        a = np.asarray(sc)
        if not np.isfinite(a).any():
            if log is not None: log["threshold_fallbacks"] += 1
            a = np.asarray([f1_score(y[:, j], (p[:, j] >= t).astype(int), zero_division=0) for t in THRESHOLD_GRID])
        out.append(float(THRESHOLD_GRID[np.where(a >= np.nanmax(a) - TIE_TOL)[0][-1]]))
    return np.asarray(out)

def oof_ml(X, y, cfg, seed, strat=None, log=None):
    px = np.zeros(y.shape) if cfg["use_xgb"] else None; pl = np.zeros(y.shape) if cfg["use_lr"] else None
    S = y if strat is None else np.c_[y, strat]
    for a, b in MultilabelStratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), S):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_ml(tf(X[a]), y[a], cfg, seed, log)
        qx, ql = proba_ml(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

# ---------------- single-label (Tokopedia) ----------------
def train_sl(X, y, cfg, seed):
    xm = lm = None
    if cfg["use_xgb"]:
        sw = None
        if cfg["cost_sensitive"]:
            cnt = np.bincount(y, minlength=K_TOK); wt = len(y) / (K_TOK * np.maximum(cnt, 1)); sw = wt[y]
        xm = xgb.XGBClassifier(num_class=K_TOK, random_state=seed, n_jobs=XGB_RUNTIME["n_jobs"],
                               **XGB_PARAMS_MULTI).fit(xgb_in(X), y, sample_weight=sw)
    if cfg["use_lr"]:
        lm = LogisticRegression(C=cfg["lr_C"], max_iter=1000, solver="lbfgs",
                                class_weight="balanced" if cfg["lr_balanced"] else None, random_state=seed).fit(X, y)
    return xm, lm

def proba_sl(X, xm, lm):
    return (xm.predict_proba(xgb_in(X)) if xm is not None else None), (lm.predict_proba(X) if lm is not None else None)

def oof_sl(X, y, cfg, seed):
    px = np.zeros((len(y), K_TOK)) if cfg["use_xgb"] else None; pl = np.zeros((len(y), K_TOK)) if cfg["use_lr"] else None
    for a, b in StratifiedKFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        tf, _, _ = fit_preprocessor(X[a], seed); xm, lm = train_sl(tf(X[a]), y[a], cfg, seed)
        qx, ql = proba_sl(tf(X[b]), xm, lm)
        if px is not None: px[b] = qx
        if pl is not None: pl[b] = ql
    return px, pl

def tune_w_sl(px, pl, y):
    best = (0.5, -np.inf)
    for w in WEIGHT_GRID:
        s = f1_score(y, np.argmax(combine(px, pl, w), 1), average="macro", zero_division=0)
        if s > best[1]: best = (float(w), s)
    return best[0]

def tune_alpha(p, y):
    # class-prior coefficients by coordinate ascent; ties within 0.005 -> value closest to 1
    alpha = np.ones(K_TOK)
    for _ in range(ALPHA_PASSES):
        for c in range(K_TOK):
            sc = []
            for a in ALPHA_GRID:
                q = alpha.copy(); q[c] = a
                sc.append(f1_score(y, np.argmax(p * q, 1), average="macro", zero_division=0))
            sc = np.asarray(sc); cand = np.where(sc >= sc.max() - TIE_TOL)[0]
            alpha[c] = ALPHA_GRID[cand[np.argmin(np.abs(ALPHA_GRID[cand] - 1))]]
    return alpha

## 5. Configurations and runners (verbatim from notebook 03)

In [ ]:
# ============================================================
# 5. CONFIGURATIONS
# ============================================================
def C(name, encoder, use_xgb=True, use_lr=True, cost_sensitive=False, lr_balanced=False, lr_C=1.0):
    return dict(name=name, encoder=encoder, use_xgb=use_xgb, use_lr=use_lr,
                cost_sensitive=cost_sensitive, lr_balanced=lr_balanced, lr_C=lr_C)

W = dict(cost_sensitive=True, lr_balanced=True)            # class weighting (both branches)
REF = "E5 Hybrid (imbalance-aware)"                        # = ME-XBAL in the old result files
CONFIGS = [
    # --- the 11 models of the manuscript (old labels: ME-XBAL (XLM-R) -> XLM-R Hybrid (imbalance-aware), ME-XBAL -> REF)
    C("XLM-R LR", "xlmr", use_xgb=False), C("XLM-R XGB", "xlmr", use_lr=False), C("XLM-R Hybrid", "xlmr"),
    C("IndoBERT LR", "indobert", use_xgb=False), C("IndoBERT XGB", "indobert", use_lr=False), C("IndoBERT Hybrid", "indobert"),
    C("E5 LR", "e5", use_xgb=False), C("E5 XGB", "e5", use_lr=False), C("E5 Hybrid", "e5"),
    C("XLM-R Hybrid (imbalance-aware)", "xlmr", **W, lr_C=0.1), C(REF, "e5", **W, lr_C=0.1),
    # --- new controls (R2-3, R1-3): 2 x 2 factorials (weighting x C) for the LR-only and the hybrid head
    C("E5 LR (C=0.1)", "e5", use_xgb=False, lr_C=0.1),
    C("E5 LR (weighted)", "e5", use_xgb=False, lr_balanced=True),
    C("E5 LR (imbalance-aware)", "e5", use_xgb=False, lr_balanced=True, lr_C=0.1),
    C("E5 XGB (cost-sensitive)", "e5", use_lr=False, cost_sensitive=True),
    C("E5 Hybrid (C=0.1)", "e5", lr_C=0.1),
    C("E5 Hybrid (weighted)", "e5", **W),
    C("XLM-R LR (imbalance-aware)", "xlmr", use_xgb=False, lr_balanced=True, lr_C=0.1),
]
CFG = {c["name"]: c for c in CONFIGS}
assert len(CFG) == len(CONFIGS)
print(len(CONFIGS), "configurations")

# ============================================================
# 6. RUNNERS
# ============================================================
def lang_onehot(groups):
    return np.eye(len(LANGS), dtype=int)[groups]

def run_sem(cfg, seed, fold_by_lang=None, w_objective=None):
    fold_by_lang = FOLD_STRATIFY_BY_LANG if fold_by_lang is None else fold_by_lang
    w_objective = w_objective or W_OBJECTIVE
    E = SEMEVAL_EMB[cfg["encoder"]]; log = new_log()
    dev = {l: np.r_[SEMEVAL_SPLITS[(l, seed)][0], SEMEVAL_SPLITS[(l, seed)][1]] for l in LANGS}
    X = np.vstack([E[l][dev[l]] for l in LANGS])
    y = np.vstack([SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[dev[l]] for l in LANGS])
    g = np.concatenate([np.full(len(dev[l]), i) for i, l in enumerate(LANGS)])
    px, pl = oof_ml(X, y, cfg, seed, lang_onehot(g) if fold_by_lang else None, log)
    w = tune_w_ml(px, pl, y, g, w_objective) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    th = tune_th(combine(px, pl, w), y, log)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_ml(tf(X), y, cfg, seed, log)
    rows, art, YT, YP = [], {"weight_xgb": w, "thresholds": th, "log": log, "n_dev": {l: len(dev[l]) for l in LANGS}}, [], []
    for l in LANGS:
        te = SEMEVAL_SPLITS[(l, seed)][2]
        yt = SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int)[te]
        pt = combine(*proba_ml(tf(E[l][te]), xm, lm), w); yp = (pt >= th[None, :]).astype(int)
        rows.append({"model": cfg["name"], "seed": seed, "lang": l, "n_test": len(te), "weight_xgb": w,
                     "macro_f1": f1_score(yt, yp, average="macro", zero_division=0)})
        art[l] = {"test_uid": SEMEVAL_DATA[l]["uid"].values[te], "y_true": yt, "y_pred": yp, "proba": pt.astype(np.float32)}
        YT.append(yt); YP.append(yp)
    YT, YP = np.vstack(YT), np.vstack(YP)
    rows.append({"model": cfg["name"], "seed": seed, "lang": "pooled", "n_test": len(YT), "weight_xgb": w,
                 "macro_f1": f1_score(YT, YP, average="macro", zero_division=0)})
    rows.append({"model": cfg["name"], "seed": seed, "lang": "average", "n_test": len(YT), "weight_xgb": w,
                 "macro_f1": float(np.mean([r["macro_f1"] for r in rows[:3]]))})
    for r in rows: r.update(constant_fits=log["constant_fits"], threshold_fallbacks=log["threshold_fallbacks"])
    return rows, art

def _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te):
    E = TOKO_EMB[cfg["encoder"]]; yt = TOKO_DF.label.values[te]
    pt = combine(*proba_sl(tf(E[te]), xm, lm), w); yp = np.argmax(pt * alpha, 1)
    row = {"model": cfg["name"], "seed": seed, "n_test": len(te), "weight_xgb": w,
           "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
           "micro_f1": f1_score(yt, yp, average="micro", zero_division=0), "accuracy": accuracy_score(yt, yp),
           "alpha": json.dumps([round(float(a), 2) for a in alpha])}
    art = {"weight_xgb": w, "alpha": alpha, "test_uid": TOKO_DF.uid.values[te], "y_true": yt, "y_pred": yp,
           "proba": pt.astype(np.float32)}
    return row, art

def run_tok(cfg, seed, fit_idx=None):
    """Fit AND select (out-of-fold) on fit_idx (default: train + validation = development partition)."""
    E = TOKO_EMB[cfg["encoder"]]; tr, va, te = TOKO_SPLITS[seed]
    dev = np.r_[tr, va] if fit_idx is None else np.asarray(fit_idx)
    X = E[dev]; y = TOKO_DF.label.values[dev]
    px, pl = oof_sl(X, y, cfg, seed)
    w = tune_w_sl(px, pl, y) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    return _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)

def run_tok_valsel(cfg, seed, fit_idx, sel_idx):
    """Fit on fit_idx only; select the ensemble weight and class priors on sel_idx (regime B)."""
    E = TOKO_EMB[cfg["encoder"]]; te = TOKO_SPLITS[seed][2]
    X = E[fit_idx]; y = TOKO_DF.label.values[fit_idx]; ys = TOKO_DF.label.values[sel_idx]
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    px, pl = proba_sl(tf(E[sel_idx]), xm, lm)
    w = tune_w_sl(px, pl, ys) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    alpha = tune_alpha(combine(px, pl, w), ys)
    return _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)

## 6. Reference results of both submissions

In [ ]:
# ============================================================
# 6. REFERENCE RESULTS OF BOTH SUBMISSIONS (per-seed scores and stored predictions)
# ============================================================
OLD_SEM, OLD_TOK = pd.read_csv(f"{OLD_DIR}/SemEval_final_raw.csv"), pd.read_csv(f"{OLD_DIR}/Tokopedia_final_raw.csv")
NEW_SEM, NEW_TOK = pd.read_csv(f"{NEW_DIR}/E1_SemEval_raw.csv"), pd.read_csv(f"{NEW_DIR}/E1_Tokopedia_raw.csv")
OLD_ENV, NEW_ENV = json.load(open(f"{OLD_DIR}/environment.json")), json.load(open(f"{NEW_DIR}/environment_cpu.json"))

OLD_PRED_DIR = os.path.join(WORK, "first_submission_predictions")
if not os.path.isdir(OLD_PRED_DIR):
    shutil.unpack_archive(f"{OLD_DIR}/predictions.zip", OLD_PRED_DIR)
OLD_PRED_DIR = next(str(p.parent) for p in Path(OLD_PRED_DIR).rglob("*.joblib"))
NEW_PRED_DIR = f"{NEW_DIR}/predictions"

def old_pred(dataset, model, seed):
    """First submission: SemEval {weight_xgb, thresholds, sun/jav/ind: {test_idx, y_true, y_pred, proba}}; Tokopedia {.., alpha, test_idx}."""
    p = f"{OLD_PRED_DIR}/{dataset}_{safe(model)}_seed{seed}.joblib"
    if not os.path.exists(p): return None
    a = joblib.load(p)
    if dataset == "SemEval":
        for l in LANGS: a[l]["test_uid"] = SEMEVAL_DATA[l]["uid"].values[a[l]["test_idx"]]
    else:
        a["test_uid"] = TOKO_DF.uid.values[a["test_idx"]]
    return a

def new_pred(dataset, model, seed):
    p = f"{NEW_PRED_DIR}/{dataset}_{safe(model)}_seed{seed}.joblib"
    return joblib.load(p) if os.path.exists(p) else None

def old_semeval_scores(model, seed):
    """Per-language Macro F1 of the first submission + average and pooled (pooled recomputed from the stored predictions)."""
    g = OLD_SEM[(OLD_SEM.model == model) & (OLD_SEM.seed == seed)].set_index("lang").macro_f1.to_dict()
    g["average"] = float(np.mean([g[l] for l in LANGS]))
    a = old_pred("SemEval", model, seed)
    if a is not None:
        g["pooled"] = f1_score(np.vstack([a[l]["y_true"] for l in LANGS]), np.vstack([a[l]["y_pred"] for l in LANGS]),
                               average="macro", zero_division=0)
        for l in LANGS:   # the stored predictions must reproduce the stored scores
            assert abs(f1_score(a[l]["y_true"], a[l]["y_pred"], average="macro", zero_division=0) - g[l]) < 1e-9, (model, seed, l)
    return g

def compare_ml(art_a, art_b):
    """Text-level comparison of two SemEval runs (same seed): aligned on test uid."""
    out = {"max_abs_proba_diff": 0.0, "pred_cells_differing": 0, "texts_differing": 0}
    for l in LANGS:
        ia = pd.Index(art_a[l]["test_uid"]); ib = pd.Index(art_b[l]["test_uid"]); assert set(ia) == set(ib), l
        o = ib.get_indexer(ia)
        dp = np.abs(np.asarray(art_a[l]["proba"], float) - np.asarray(art_b[l]["proba"], float)[o])
        dy = np.asarray(art_a[l]["y_pred"]) != np.asarray(art_b[l]["y_pred"])[o]
        out["max_abs_proba_diff"] = max(out["max_abs_proba_diff"], float(dp.max()))
        out["pred_cells_differing"] += int(dy.sum()); out["texts_differing"] += int(dy.any(1).sum())
    out["thresholds_identical"] = bool(np.allclose(np.asarray(art_a["thresholds"], float), np.asarray(art_b["thresholds"], float)))
    return out

def compare_sl(art_a, art_b):
    ia = pd.Index(art_a["test_uid"]); ib = pd.Index(art_b["test_uid"]); assert set(ia) == set(ib)
    o = ib.get_indexer(ia)
    return {"max_abs_proba_diff": float(np.abs(np.asarray(art_a["proba"], float) - np.asarray(art_b["proba"], float)[o]).max()),
            "texts_differing": int((np.asarray(art_a["y_pred"]) != np.asarray(art_b["y_pred"])[o]).sum()),
            "alpha_identical": bool(np.allclose(np.asarray(art_a["alpha"], float), np.asarray(art_b["alpha"], float)))}

print("First submission environment:", {k: OLD_ENV.get(k) for k in ["python", "torch", "transformers", "sklearn", "xgboost", "numpy", "gpu"]})
print("Revision environment        :", {k: NEW_ENV.get(k) for k in ["python", "torch", "transformers", "sklearn", "xgboost", "numpy", "cpu_count", "fold_stratify_by_lang"]})
print("This session                :", {"python": platform.python_version(), "sklearn": sklearn.__version__, "xgboost": xgb.__version__,
                                         "numpy": np.__version__, "cpu": cpu_info().replace(chr(10), " | ")})


## 7. Block B — SemEval LR-only under both fold procedures

In [ ]:
# ============================================================
# 7. BLOCK B — SemEval LR-only under both fold procedures, compared text by text with both submissions
# ============================================================
B_RUNS, B_CMP = f"{OUT_ROOT}/B_semeval_runs.csv", f"{OUT_ROOT}/B_semeval_vs_reference.csv"
if RUN_SEMEVAL:
    done = done_keys(B_RUNS, ["procedure", "model", "seed"])
    for proc, by_lang in PROCEDURES.items():
        for name in LR_MODELS:
            for seed in tqdm(SEEDS, desc=f"{proc} | {name}"):
                if (proc, name, str(seed)) in done: continue
                rows, art = run_sem(CFG[name], seed, fold_by_lang=by_lang, w_objective="pooled")
                append_csv(B_RUNS, [dict(r, procedure=proc) for r in rows])
                joblib.dump(art, f"{OUT_ROOT}/predictions/SemEval_{safe(name)}_{proc}_seed{seed}.joblib", compress=3)
                cmp = []
                for ref_tag, ref_art, ref_scores in [
                        ("first submission", old_pred("SemEval", name, seed), old_semeval_scores(name, seed)),
                        ("revision", new_pred("SemEval", name, seed),
                         NEW_SEM[(NEW_SEM.model == name) & (NEW_SEM.seed == seed)].set_index("lang").macro_f1.to_dict())]:
                    if ref_art is None: continue
                    this = {r["lang"]: r["macro_f1"] for r in rows}
                    cmp.append({"procedure": proc, "model": name, "seed": seed, "reference": ref_tag,
                                **{f"dF1_{l}": this[l] - ref_scores[l] for l in ["sun", "jav", "ind", "average", "pooled"] if l in ref_scores},
                                "scores_identical": all(abs(this[l] - ref_scores[l]) < 1e-9 for l in LANGS),
                                **compare_ml(art, ref_art)})
                append_csv(B_CMP, cmp)

if os.path.exists(B_CMP):
    c = pd.read_csv(B_CMP)
    summ = (c.groupby(["procedure", "model", "reference"])
             .agg(seeds=("seed", "nunique"), seeds_scores_identical=("scores_identical", "sum"),
                  max_abs_proba_diff=("max_abs_proba_diff", "max"), texts_differing=("texts_differing", "sum"),
                  seeds_thresholds_identical=("thresholds_identical", "sum"), mean_dF1_average=("dF1_average", "mean"))
             .reset_index())
    summ.to_csv(f"{OUT_ROOT}/B_semeval_summary.csv", index=False)
    print(summ.to_string(index=False))


## 8. Block C — Tokopedia LR-only and mismatch diagnosis

In [ ]:
# ============================================================
# 8. BLOCK C — Tokopedia LR-only, run twice in this session, compared with both submissions + mismatch diagnosis
# ============================================================
def run_tok_diag(cfg, seed):
    """run_tok of notebook 03 that also returns the out-of-fold probabilities used to select the class priors."""
    E = TOKO_EMB[cfg["encoder"]]; tr, va, te = TOKO_SPLITS[seed]
    dev = np.r_[tr, va]; X = E[dev]; y = TOKO_DF.label.values[dev]
    px, pl = oof_sl(X, y, cfg, seed)
    w = tune_w_sl(px, pl, y) if (cfg["use_xgb"] and cfg["use_lr"]) else (1.0 if cfg["use_xgb"] else 0.0)
    p_oof = combine(px, pl, w); alpha = tune_alpha(p_oof, y)
    tf, _, _ = fit_preprocessor(X, seed); xm, lm = train_sl(tf(X), y, cfg, seed)
    row, art = _tok_eval(cfg, seed, xm, lm, tf, w, alpha, te)
    return row, art, p_oof, y

def diagnose(ref_art, art, p_oof, y_oof):
    """Why does this run differ from a reference run with the same splits and embeddings?"""
    a_this, a_ref = np.asarray(art["alpha"], float), np.asarray(ref_art["alpha"], float)
    f_this = f1_score(y_oof, np.argmax(p_oof * a_this, 1), average="macro", zero_division=0)
    f_ref  = f1_score(y_oof, np.argmax(p_oof * a_ref, 1), average="macro", zero_division=0)
    cmp = compare_sl(art, ref_art)
    if cmp["texts_differing"] == 0: cause = "identical predictions"
    elif not cmp["alpha_identical"] and abs(f_this - f_ref) <= TIE_TOL:
        cause = "class-prior search: both coefficient vectors lie within the tie tolerance on the out-of-fold data"
    elif not cmp["alpha_identical"]:
        cause = "class-prior search: different coefficients (not a tie on this run's out-of-fold data)"
    else:
        cause = "same coefficients; test probabilities differ at the decision boundary"
    return {"alpha_this": json.dumps([round(float(v), 2) for v in a_this]), "alpha_reference": json.dumps([round(float(v), 2) for v in a_ref]),
            "oof_f1_with_this_alpha": f_this, "oof_f1_with_reference_alpha": f_ref, "cause": cause, **cmp}

C_RUNS, C_CMP = f"{OUT_ROOT}/C_tokopedia_runs.csv", f"{OUT_ROOT}/C_tokopedia_vs_reference.csv"
if RUN_TOKOPEDIA:
    done = done_keys(C_RUNS, ["model", "seed"])
    for name in LR_MODELS:
        for seed in tqdm(SEEDS, desc=f"Tokopedia | {name}"):
            if (name, str(seed)) in done: continue
            reps = [run_tok_diag(CFG[name], seed) for _ in range(TOK_REPEATS)]
            row, art, p_oof, y_oof = reps[0]
            same = all(np.array_equal(r[1]["y_pred"], art["y_pred"]) and np.array_equal(r[1]["proba"], art["proba"]) for r in reps[1:])
            append_csv(C_RUNS, [dict(row, repeats=TOK_REPEATS, repeats_identical=same)])
            joblib.dump(art, f"{OUT_ROOT}/predictions/Tokopedia_{safe(name)}_seed{seed}.joblib", compress=3)
            cmp = []
            for ref_tag, ref_art, ref_df in [("first submission", old_pred("Tokopedia", name, seed), OLD_TOK),
                                             ("revision", new_pred("Tokopedia", name, seed), NEW_TOK)]:
                if ref_art is None: continue
                ref_f1 = float(ref_df[(ref_df.model == name) & (ref_df.seed == seed)].macro_f1.iloc[0])
                cmp.append({"model": name, "seed": seed, "reference": ref_tag, "macro_f1_this": row["macro_f1"],
                            "macro_f1_reference": ref_f1, "dF1": row["macro_f1"] - ref_f1,
                            **diagnose(ref_art, art, p_oof, y_oof)})
            append_csv(C_CMP, cmp)

if os.path.exists(C_CMP):
    c = pd.read_csv(C_CMP)
    print(c.groupby(["model", "reference"]).agg(seeds=("seed", "nunique"), identical=("texts_differing", lambda x: int((x == 0).sum())),
                                                max_abs_dF1=("dF1", lambda x: float(np.abs(x).max())),
                                                max_abs_proba_diff=("max_abs_proba_diff", "max")).to_string())
    print(c[c.texts_differing > 0][["model", "seed", "reference", "dF1", "texts_differing", "max_abs_proba_diff",
                                    "alpha_this", "alpha_reference", "cause"]].to_string(index=False))
    print("Repeats identical in this session:", bool(pd.read_csv(C_RUNS).repeats_identical.all()))


## 9. Block D — Table 3 reconciliation

In [ ]:
# ============================================================
# 9. BLOCK D — cell-by-cell reconciliation of Table 3 (LR rows) + per-seed table
# ============================================================
COLS = ["sun", "jav", "ind", "average", "pooled"]
b = pd.read_csv(B_RUNS)
per_seed = []
for name in LR_MODELS:
    for seed in SEEDS:
        o = old_semeval_scores(name, seed)
        n = NEW_SEM[(NEW_SEM.model == name) & (NEW_SEM.seed == seed)].set_index("lang").macro_f1
        for l in COLS:
            r = {"model": name, "seed": seed, "lang": l, "first_submission": o.get(l, np.nan), "revision": n.get(l, np.nan)}
            for proc in PROCEDURES:
                g = b[(b.procedure == proc) & (b.model == name) & (b.seed == seed) & (b.lang == l)]
                r[f"this_run_{proc}"] = float(g.macro_f1.iloc[0]) if len(g) else np.nan
            per_seed.append(r)
PS = pd.DataFrame(per_seed); PS.to_csv(f"{OUT_ROOT}/D_table3_LR_per_seed.csv", index=False)

fmt = lambda v: f"{np.mean(v):.4f} ± {np.std(v, ddof=1):.4f}"
rows = []
for name in LR_MODELS:
    for src, col in [("First submission (Table 3)", "first_submission"),
                     ("This run, first-submission folds (emotions only)", "this_run_first_submission_folds"),
                     ("Revision (Table 3)", "revision"),
                     ("This run, revision folds (emotions + language)", "this_run_revision_folds")]:
        g = PS[PS.model == name]
        rows.append({"model": name, "source": src, **{l: fmt(g[g.lang == l][col]) for l in COLS}})
    g = PS[PS.model == name]
    for a, c, lab in [("this_run_first_submission_folds", "first_submission", "identical to first submission (seed×language cells)"),
                      ("this_run_revision_folds", "revision", "identical to revision (seed×language cells)")]:
        rows.append({"model": name, "source": lab,
                     **{l: f"{int((np.abs(g[g.lang == l][a] - g[g.lang == l][c]) < 1e-9).sum())}/{len(SEEDS)}" for l in COLS}})
REC = pd.DataFrame(rows); REC.to_csv(f"{OUT_ROOT}/D_table3_LR_reconciliation.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
print(REC.to_string(index=False))

# short machine-written summary for the response letter
cB = pd.read_csv(B_CMP); cC = pd.read_csv(C_CMP) if os.path.exists(C_CMP) else pd.DataFrame()
lines = [f"# Reconciliation summary ({time.strftime('%Y-%m-%d %H:%M')})", "",
         f"Reference: {REPO_SOURCE}. Splits identical to both submissions: {bool(SPLIT_CHECK.identical.all())}.", ""]
for proc in PROCEDURES:
    for ref in ["first submission", "revision"]:
        g = cB[(cB.procedure == proc) & (cB.reference == ref)]
        if len(g):
            lines.append(f"- SemEval, procedure `{proc}` vs {ref}: {int(g.scores_identical.sum())}/{len(g)} model×seed runs with identical "
                         f"per-language scores; max |Δp| = {g.max_abs_proba_diff.max():.2e}; texts with a different label set = {int(g.texts_differing.sum())}.")
if len(cC):
    for ref in ["first submission", "revision"]:
        g = cC[cC.reference == ref]
        lines.append(f"- Tokopedia vs {ref}: {int((g.texts_differing == 0).sum())}/{len(g)} model×seed runs identical; "
                     f"max |ΔF1| = {np.abs(g.dF1).max():.4f}; causes of the remaining differences: "
                     + "; ".join(f"{k} ({v})" for k, v in g[g.texts_differing > 0].cause.value_counts().items()))
open(f"{OUT_ROOT}/D_summary.md", "w").write("\n".join(lines) + "\n"); print("\n".join(lines))


## 10. Environment record and archive

In [ ]:
# ============================================================
# 10. ENVIRONMENT + ZIP
# ============================================================
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgb.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "cpu": cpu_info(), "cpu_count": os.cpu_count(), "reference": REPO_SOURCE, "reference_commit": REF_COMMIT,
       "procedures": PROCEDURES, "lr_models": LR_MODELS, "seeds": SEEDS, "smoke_test": SMOKE_TEST,
       "embedding_cache_input": CACHE_IN, "embedding_fingerprints": EMB_FINGERPRINT,
       "first_submission_environment": OLD_ENV, "revision_environment": {k: v for k, v in NEW_ENV.items() if k != "embedding_fingerprints"},
       "runtime_min": round((time.time() - NOTEBOOK_T0) / 60, 1)}
json.dump(env, open(f"{OUT_ROOT}/environment_reconcile.json", "w"), indent=2, default=str)
try:
    open(f"{OUT_ROOT}/pip_freeze_reconcile.txt", "w").write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
except Exception as e: print("pip freeze failed:", e)
shutil.make_archive(f"{WORK}/reconcile_lr_outputs", "zip", OUT_ROOT)
print("Archive:", f"{WORK}/reconcile_lr_outputs.zip", "| runtime", env["runtime_min"], "min")
